# 5교시 · 평가 · 어댑터 내보내기

> **VLM 경량화 2일 과정 · Day 1 (5교시) · 실습**
> 실습 환경: **Google Colab (T4 GPU)** · 모델: **Qwen/Qwen3-VL-4B-Instruct** · 데이터: **HuggingFaceM4/ChartQA**

---

## 이 교시의 목표
- **Relaxed Accuracy**(ChartQA 표준 지표)를 직접 구현한다.
- 학습 **전/후** val 점수를 비교해 파인튜닝 효과를 정량화한다.
- 학습한 **LoRA 어댑터(≈33MB)를 HF Hub(비공개)에 push**해 Day2(RunPod)로 넘긴다.
- 무거운 **병합(fp16 ≈8.3GB)은 Day2(A100)** 에서 AWQ 직전에 수행한다 — 코랩 RAM 한계를 피한다.


## 0. 공통 헤더 — Google Drive(VLM_FT_2) 마운트 + HF_TOKEN 로드

> 📌 **모든 Day 1 노트북은 이 셀을 먼저 실행합니다.** Google Drive의 작업 폴더 **`VLM_FT_2`** 를 마운트하고, `.env`의 **HF_TOKEN**을 불러옵니다.
> - `VLM_DIR` / `DATA_DIR` : 교시 간 공유 폴더(전처리 데이터·어댑터·결과가 여기 모입니다).
> - **HF_TOKEN**: `VLM_FT_2/.env` 에 `HF_TOKEN=hf_...` 한 줄을 넣어두면 자동 로드됩니다(다운로드 경고 방지·비공개 모델 접근). `login()`은 호출하지 않습니다(환경변수만으로 충분, 경고 방지).

In [1]:
# ════════════════════════════════════════════════════════════
#  공통 헤더 · Google Drive(VLM_FT_2) 마운트 + HF_TOKEN(.env) 로드
#  (모든 Day1 노트북 상단에서 동일하게 실행)
# ════════════════════════════════════════════════════════════
import os

# (1) Google Drive 마운트 → 작업 폴더 VLM_FT_2 (교시 간 데이터·어댑터·결과 공유)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    VLM_DIR = '/content/drive/MyDrive/VLM_FT_2'      # Drive 경로(권장)
except Exception:
    VLM_DIR = '/content/VLM_FT_2'                     # Colab 아니면 로컬 폴백
DATA_DIR = f'{VLM_DIR}/data'
os.makedirs(DATA_DIR, exist_ok=True)

# (2) .env 에서 HF_TOKEN 로드. login()은 부르지 않음(환경변수만으로 인증, 경고 없음).
try:
    from dotenv import load_dotenv
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'python-dotenv'])
    from dotenv import load_dotenv
ENV_PATH = f'{VLM_DIR}/.env'
if os.path.exists(ENV_PATH):
    load_dotenv(ENV_PATH)
    print('HF_TOKEN:', '로드됨' if os.environ.get('HF_TOKEN') else '없음')
else:
    print(f'.env 없음 — {ENV_PATH} 에 HF_TOKEN=hf_... 한 줄을 만들면 자동 로드됩니다(공개 데이터만 쓰면 없어도 됨)')
print('작업 폴더 VLM_DIR =', VLM_DIR)

Mounted at /content/drive
HF_TOKEN: 로드됨
작업 폴더 VLM_DIR = /content/drive/MyDrive/VLM_FT_2


## 0. 준비 — 설치·프로세서·평가셋

In [2]:
!pip install -q -U "transformers>=4.57" "accelerate>=1.0" "bitsandbytes>=0.44" "datasets>=3.0" "peft>=0.13" "huggingface_hub>=0.25" "qwen-vl-utils>=0.0.8" "python-dotenv" "pillow<12"
import torch, transformers
print('transformers:', transformers.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 112.5 MB/s eta 0:00:0000:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.1/721.1 kB 49.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 13.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.4/35.4 MB 63.4 MB/s eta 0:00:00:00:0100:01
transformers: 5.12.1


In [3]:
# 프로세서 + 평가용 val 서브셋. VLM_DIR/DATA_DIR/os 는 상단 공통 헤더가 설정.
from transformers import AutoProcessor
from datasets import load_dataset, load_from_disk

# 모델 ID 및 이미지 크기 범위 설정 (VLM 입력 제약)
MODEL_ID = 'Qwen/Qwen3-VL-4B-Instruct'
MIN_PIXELS, MAX_PIXELS = 64 * 28 * 28, 256 * 28 * 28

# 프로세서 로드: 이미지·텍스트를 모델 입력 형식으로 정리
processor = AutoProcessor.from_pretrained(MODEL_ID, min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS)
processor.tokenizer.padding_side = 'right'  # 토큰 패딩을 오른쪽에 적용

# 평가셋 설정: Day1-3에서 저장한 동일한 300개 샘플 사용 (교시 간 일치성 확보)
SEED, VAL_N = 42, 300   # 난수 시드 및 val 샘플 수
VAL_PATH = f'{DATA_DIR}/chartqa_val_{VAL_N}'

# Day1-3 저장본이 있으면 로드, 없으면 동일 시드로 샘플링(재현성 보장)
if os.path.isdir(VAL_PATH):
    val_set = load_from_disk(VAL_PATH)
    print('Day1-3 저장본 로드:', VAL_PATH)
else:
    val_set = load_dataset('HuggingFaceM4/ChartQA', split='val').shuffle(seed=SEED).select(range(VAL_N))
    print('저장본 없음 → 동일 시드 샘플링')

EVAL_N = len(val_set)
print('평가 샘플 수:', len(val_set))

preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/5.50k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.50k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/10.9k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

Day1-3 저장본 로드: /content/drive/MyDrive/VLM_FT_2/data/chartqa_val_300
평가 샘플 수: 300


### 🔍 결과 해석 — 평가셋 준비
- `Day1-3 저장본 로드 ... chartqa_val_300` / `평가 샘플 수: 300` → 3교시에서 저장한 **동일한 val 300개**를 그대로 씁니다(교시 간 일치). 시드가 같아 누구나 같은 셋으로 평가합니다.

## 1. Relaxed Accuracy — ChartQA의 표준 지표

차트 질문의 답은 대부분 **숫자**입니다. 정답이 `95`인데 모델이 `94.8`을 냈다면 *거의 맞은* 것이죠. 그래서 ChartQA는 **숫자 답은 ±5% 오차까지 정답**으로 인정합니다(Relaxed = 느슨한).

- **숫자 답**: `|예측 − 정답| / |정답| ≤ 0.05` 이면 정답.
- **텍스트 답**(Yes/No, 연도 라벨 등): **대소문자 무시 완전일치**.

> ⚠️ 학습 **전** 모델은 답을 장황하게 말해(예: "The value is about 95.") 숫자 파싱이 실패 → 점수가 낮게 나옵니다. 이건 버그가 아니라 **"형식을 안 지킨다"는 사실**이고, 파인튜닝이 고치려는 바로 그 문제입니다.

In [4]:
# ── Relaxed Accuracy 구현 ────────────────────────────────────

def _to_number(s: str):
    """
    문자열을 숫자(float)로 변환.
    예: '12,300' -> 12300.0, '47%' -> 47.0, '95.0' -> 95.0
    변환 실패 시 None 반환.
    """
    # 공백 제거 -> 천 단위 콤마 제거 -> % 제거 -> 다시 공백 정리
    s = s.strip().replace(',', '').replace('%', '').strip()
    try:
        return float(s)
    except ValueError:
        # 숫자로 해석할 수 없는 문자열
        return None


def relaxed_match(pred: str, gold: str, tol: float = 0.05) -> bool:
    """
    ChartQA Relaxed Accuracy 규칙으로 정오 판단.
    - 둘 다 숫자면: 상대 오차 <= tol(기본 5%)면 정답
    - 숫자가 아니면: 대소문자 무시 완전 일치
    """
    # 앞뒤 공백 정리
    pred, gold = pred.strip(), gold.strip()

    # 숫자 파싱 시도
    pn, gn = _to_number(pred), _to_number(gold)

    if pn is not None and gn is not None:  # 둘 다 숫자인 경우
        if gn == 0:
            # 정답이 0일 때는 0으로 정확히 맞춰야 정답
            return pn == 0
        # 상대 오차( |예측-정답| / |정답| )가 허용 범위 이내면 정답
        return abs(pn - gn) / abs(gn) <= tol

    # 텍스트 정답은 대소문자 무시 후 완전 일치 비교
    return pred.lower() == gold.lower()


# ── 간단 자가검증 ─────────────────────────────────────────────
assert relaxed_match('95', '95')                      # 정확히 동일한 숫자
assert relaxed_match('94.8', '95')                    # 0.2% 오차 -> 정답
assert not relaxed_match('80', '95')                  # 약 15.8% 오차 -> 오답
assert relaxed_match('yes', 'Yes')                    # 대소문자만 다름 -> 정답
assert not relaxed_match('The value is 95', '95')     # 장황한 문장 -> 숫자 파싱 실패 -> 오답

print('relaxed_match 자가검증 통과')

relaxed_match 자가검증 통과


## 2. 학습 전(baseline) 평가

In [5]:
# ── 4bit base 모델 로드 + 추론/평가 함수 정의 ─────────────────────────────
# 주의: torch, MODEL_ID, processor, val_set, relaxed_match 는 이전 셀에서 이미 준비됨
from transformers import Qwen3VLForConditionalGeneration, BitsAndBytesConfig

# 4bit 양자화 설정 (T4 메모리 절약용 QLoRA 추론 세팅)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                    # 가중치를 4bit로 로드
    bnb_4bit_quant_type='nf4',            # 4bit 양자화 타입(NF4)
    bnb_4bit_use_double_quant=True,       # 이중 양자화로 메모리 추가 절약
    bnb_4bit_compute_dtype=torch.float16, # 연산은 fp16으로 수행
)

# 베이스 모델 로드 (자동 device_map으로 GPU/메모리에 맞춰 배치)
base_model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    dtype=torch.float16,
    attn_implementation='sdpa',           # 안정적이고 빠른 attention 구현
).eval()                                  # 평가 모드 (dropout 비활성화)

# ── 시스템 프롬프트 + 메시지/이미지 헬퍼 ───────────────────────────────
from PIL import Image
from qwen_vl_utils import process_vision_info  # 메시지에서 이미지 입력 텐서 추출

# 답변 형식을 짧게 강제해 장황한 출력(숫자 파싱 실패)을 줄이기 위한 시스템 프롬프트
SYSTEM = "당신은 차트를 읽고 질문에 답하는 도우미입니다. 설명 없이 한 단어 또는 숫자로만 짧게 답하세요."

def shrink(img, max_side=768):
    """
    긴 변 기준으로 최대 크기를 제한해 추론 메모리/속도를 안정화.
    - 비율 유지 축소
    - 결과 이미지는 RGB로 통일
    """
    w, h = img.size
    s = max_side / max(w, h)
    if s < 1:  # 원본이 max_side보다 큰 경우에만 축소
        img = img.resize((int(w * s), int(h * s)), Image.LANCZOS)
    return img.convert("RGB")  # grayscale/palette 등도 RGB로 정규화

def to_messages(image, question, answer=None):
    """
    Qwen-VL 채팅 포맷 메시지 생성.
    - system: 출력 스타일 규칙
    - user: 이미지 + 질문
    - assistant: answer가 있을 때만 추가(학습용 포맷)
    """
    msgs = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM}]},
        {"role": "user", "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": question}
        ]},
    ]
    if answer is not None:
        msgs.append({"role": "assistant", "content": [{"type": "text", "text": answer}]})
    return msgs

@torch.no_grad()
def predict(model, image, question, max_new_tokens=32):
    """
    단일 샘플 추론:
    1) 메시지 구성
    2) chat template 적용
    3) vision/text 텐서화
    4) 생성 결과에서 입력 프롬프트 부분을 제외하고 디코딩
    """
    msgs = to_messages(shrink(image), question, None)  # 추론 시 정답(answer)은 넣지 않음
    text = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    img_inputs, _ = process_vision_info(msgs)

    # processor 출력 텐서를 모델 디바이스로 이동
    inputs = processor(text=[text], images=img_inputs, return_tensors="pt").to(model.device)

    # 결정적 추론(do_sample=False)으로 평가 재현성 확보
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)

    # 생성 시퀀스에서 입력 길이 이후 토큰만 잘라 실제 답변만 추출
    gen = out[0][inputs["input_ids"].shape[1]:]
    return processor.decode(gen, skip_special_tokens=True).strip()

def evaluate(model, dataset):
    """
    데이터셋 전체 Relaxed Accuracy 계산.
    - 숫자/텍스트 판정 규칙은 relaxed_match 함수(이전 셀 구현)를 사용
    """
    correct = 0
    for ex in dataset:
        pred = predict(model, ex['image'], ex['query'])
        gold = str(ex['label'][0])  # ChartQA label 포맷 정규화
        if relaxed_match(pred, gold):
            correct += 1
    return correct / len(dataset)

# 학습 전(base) 성능 측정
acc_before = evaluate(base_model, val_set)
print(f'학습 전 Relaxed Accuracy: {acc_before:.3f}')

model.safetensors.index.json:   0%|          | 0.00/64.7k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/713 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


학습 전 Relaxed Accuracy: 0.730


### 🔍 결과 해석 — 학습 전(baseline) 점수
- `학습 전 Relaxed Accuracy: 0.730` → **베이스 4B가 이미 73%** 입니다. Qwen3-VL-4B가 원래 차트를 꽤 잘 읽고, **여기 baseline에도 시스템 프롬프트(짧게 답하기)를 적용**하기 때문에 형식 이점까지 이미 갖춘 상태입니다. (Day1-2의 '맨 베이스라인'은 시스템 프롬프트가 없어 장황했던 것과 대비됩니다.)
- 중간의 `bitsandbytes ... FutureWarning`은 4bit 로드 시 뜨는 **정상 경고**입니다.
- 시사점: **출발점이 높으면 올릴 여지가 작습니다** — 아래 전/후 비교에서 이 점이 드러납니다.

## 3. 학습 후 평가 — 어댑터 부착

4교시에서 저장한 LoRA 어댑터를 base 위에 얹어 다시 평가합니다.

In [6]:
# ── 어댑터 로드 + 학습 후 평가 ───────────────────────────────
import os
from peft import PeftModel

# 4교시에서 저장한 LoRA 어댑터 경로
# (VLM_DIR은 공통 헤더 셀에서 이미 설정됨)
ADAPTER_DIR = f'{VLM_DIR}/lora_adapter'

# 어댑터 폴더가 실제로 존재하는지 확인
# 없으면 즉시 에러를 내어 이후 셀에서 원인 추적이 쉬워짐
assert os.path.isdir(ADAPTER_DIR), f'어댑터 폴더 없음: {ADAPTER_DIR}'

# 이미 로드된 4bit 베이스 모델(base_model)에
# LoRA 어댑터를 부착한 평가용 모델 생성
model_after = PeftModel.from_pretrained(base_model, ADAPTER_DIR).eval()

# 동일한 검증셋(val_set)으로 학습 후 성능 측정
acc_after = evaluate(model_after, val_set)

# 학습 후 Relaxed Accuracy 출력
print(f'학습 후 Relaxed Accuracy: {acc_after:.3f}')

학습 후 Relaxed Accuracy: 0.733


### 🔍 결과 해석 — 학습 후 점수
- `학습 후 Relaxed Accuracy: 0.733` → 73.0% → 73.3%. **같은 평가셋·같은 프롬프트**로 공정 비교한 결과입니다(전/후 모두 시스템 프롬프트 사용). 변화량 해석은 다음 셀에서.

In [ ]:
# ── 전/후 비교표 + 정성 샘플 ─────────────────────────────────
# W' = Wg(2GB) + △W(150MB) 병합모델이 아니라 로라어뎁터를 붙인거라 모델 로드 시 시간 메모리 쬐끔
# 표 헤더 출력: 왼쪽은 항목명, 오른쪽은 Relaxed Accuracy 값
print(f'{"":<10}{"Relaxed Acc":>12}')
print('-' * 22)

# 학습 전/후 점수와 개선폭(차이) 출력
print(f'{"학습 전":<8}{acc_before:>12.3f}')                 # baseline 점수
print(f'{"학습 후":<8}{acc_after:>12.3f}')                  # LoRA 적용 후 점수
print(f'{"개선":<8}{acc_after - acc_before:>+12.3f}')       # 후 - 전 (증감 부호 포함)
print('=' * 50)

# 같은 검증 샘플 5개를 뽑아 학습 전/후 예측을 정성적으로 비교
for i in range(5):
    ex = val_set[i]                                          # i번째 검증 예시 (이미지/질문/정답)
    p_before = predict(base_model, ex['image'], ex['query']) # 베이스 모델 예측
    p_after  = predict(model_after, ex['image'], ex['query'])# 어댑터 적용 모델 예측

    # 질문, 정답, 전/후 예측을 나란히 출력해 변화 확인
    print(f"Q: {ex['query']}")
    print(f"  정답  : {ex['label'][0]!r}")
    print(f"  학습전: {p_before!r}")
    print(f"  학습후: {p_after!r}")
    print('-' * 50)

           Relaxed Acc
----------------------
학습 전           0.730
학습 후           0.733
개선            +0.003
Q: What was the value of the pizza delivery market in the UK at the end of 2017?
  정답  : '6.2'
  학습전: '2.1'
  학습후: '2.1'
--------------------------------------------------
Q: What is the smallest value represented
  정답  : '9'
  학습전: '1'
  학습후: '1'
--------------------------------------------------
Q: Is Croatia global hunger index extremely alarminhg?
  정답  : 'No'
  학습전: 'No'
  학습후: 'No'
--------------------------------------------------
Q: What was Russia's fertility rate in 2020?
  정답  : '1.74'
  학습전: '1.74'
  학습후: '1.74'
--------------------------------------------------
Q: What is the ratio of fixed broadband in the years 2018 and 2019?
  정답  : '1'
  학습전: '1'
  학습후: '1'
--------------------------------------------------


### vLLM - 모델 서빙 엔진 사용하는 이유
- 멀티 어뎁터 지원
- KVCache (Paged Attention) 제공
- Continouse Bettery 

### 🔍 결과 해석 — 전/후 비교 (정직하게 읽기)
- **개선 +0.003** — 거의 측정 노이즈 수준입니다. 정성 샘플도 학습 전/후 예측이 **모두 동일**합니다(`2.1`·`1`·`No`·`1.74`·`99`). 즉 이번 LoRA는 출력을 거의 바꾸지 못했습니다.
- **왜 이렇게 작나?** ① **출발점이 높음**(0.730) — 강한 베이스 + baseline에도 시스템 프롬프트가 적용돼 *형식 이점이 이미 baseline에 반영*됩니다 → 순수 정확도 변화만 남습니다. ② **아주 짧은 학습**(3,000샘플·1 epoch·375스텝·r=16). 형식은 빠르게 잡히지만 *정밀한 수치 판독*은 이 정도로는 잘 개선되지 않습니다.
- 맞고 틀림의 패턴: 쉬운 것(`No`, `1.74`)은 전/후 모두 맞고, **수치 읽기**(`6.2`를 `2.1`로, 비율을 `99`로)는 전/후 모두 틀립니다 — 어려운 판독은 더 많은 학습이 필요합니다.
- **올리려면**: epoch·데이터 수·LoRA rank(r)를 키우면 됩니다. (참고: baseline을 *시스템 프롬프트 없이* 측정하면 형식 이득까지 합쳐져 격차가 커 보이지만, 이번 과정은 **형식을 양쪽 동일하게 두고 순수 정확도를 보는 공정 비교**를 택했습니다.)

> 💡 이건 실패가 아니라 **평가를 정직하게 하는 법**을 보여주는 결과입니다 — *"베이스가 이미 잘하면, 소량 데이터 LoRA의 효과는 작게 측정된다."*

## 4. LoRA 어댑터를 Hub에 push (→ Day2에서 병합 + AWQ)

병합·저장은 **무료 코랩 RAM에 부담**(fp16 4B ≈ 8.3GB)이라, Day1에서는 **가벼운 어댑터(≈33MB)만** Hub에 올립니다. 실제 **병합은 Day2(RunPod A100·대용량)** 에서 AWQ 직전에 수행합니다 — 메모리도 넉넉하고 *"병합본을 또 받는"* 왕복도 사라집니다.

In [ ]:
# ── LoRA 어댑터를 Hub에 push (비공개) — 33MB라 코랩 RAM 부담 없음 ──
# 허깅페이스에 private에 업로드하고 https://huggingface.co/leeunzin 확인
import os  # 환경변수(HF_TOKEN) 확인, 로컬 폴더 목록 확인에 사용

# HF_TOKEN이 아직 환경변수에 없다면, 사용자 입력으로 받아 .env 파일에 저장
if not os.environ.get('HF_TOKEN'):
    from getpass import getpass      # 토큰 입력 시 화면에 값이 노출되지 않도록 처리
    from dotenv import load_dotenv   # .env 파일을 환경변수로 로드

    tok = getpass('HF write 토큰 입력(최초 1회): ').strip()  # 공백 제거
    with open(ENV_PATH, 'w') as f:                           # 공통 헤더에서 정의한 ENV_PATH에 저장
        f.write(f'HF_TOKEN={tok}\n')                         # HF_TOKEN 한 줄 작성
    load_dotenv(ENV_PATH)                                    # 방금 저장한 .env를 즉시 반영
    print('.env 생성:', ENV_PATH)

# 최종적으로 HF_TOKEN이 반드시 있어야 업로드 가능
assert os.environ.get('HF_TOKEN'), '.env 에 HF_TOKEN 이 없습니다'

from huggingface_hub import HfApi  # Hugging Face Hub API 클라이언트

# 업로드 대상 리포지토리(모델 리포) 이름
ADAPTER_REPO = 'leeunzin/Qwen3-VL-4B-ChartQA-lora'   # 본인 계정으로 사용 시 이 형식

api = HfApi()  # Hub API 객체 생성

# 리포지토리가 없으면 생성, 이미 있으면 그대로 사용
api.create_repo(ADAPTER_REPO, private=True, exist_ok=True)

# Day1-4에서 저장한 어댑터 폴더 전체 업로드
# (adapter_model.safetensors, adapter_config.json, processor/tokenizer 파일 등)
api.upload_folder(
    folder_path=ADAPTER_DIR,   # 공통 셀/이전 셀에서 정의된 로컬 어댑터 폴더
    repo_id=ADAPTER_REPO,      # 업로드 대상 Hub 리포
    repo_type='model'          # 모델 리포지토리 타입
)

# 업로드 결과 확인 로그
print('어댑터 업로드 완료:', ADAPTER_REPO)
print('업로드 내용:', os.listdir(ADAPTER_DIR))  # 로컬 폴더 파일 목록 출력

어댑터 업로드 완료: leeunzin/Qwen3-VL-4B-ChartQA-lora
업로드 내용: ['README.md', 'adapter_model.safetensors', 'adapter_config.json', 'chat_template.jinja', 'tokenizer_config.json', 'tokenizer.json', 'processor_config.json']


### 🔍 결과 해석 — 어댑터 업로드
- `어댑터 업로드 완료: <당신의 계정>/Qwen3-VL-4B-ChartQA-lora` → **어댑터(약 33MB)만** Hub 비공개로 올라갔습니다. 파일에 `adapter_model.safetensors`(LoRA 가중치)·`adapter_config.json`·프로세서/토크나이저가 들어 있어, Day2가 **base 4B에 이 어댑터를 붙여 병합**할 수 있습니다.
- 무거운 fp16 병합(≈8.3GB)은 **Day2(RunPod A100)** 에서 수행하므로 코랩 RAM 문제가 없습니다.

## 6. 정리 + 다음 교시 예고

- **Relaxed Accuracy**로 학습 전/후를 비교했습니다. 점수 상승과 함께 *답이 짧고 형식을 지키는* 방향으로 바뀐 것을 정성 샘플에서 확인했습니다.
- 학습한 **LoRA 어댑터를 Hub(비공개)에 올려** Day2 이관을 마쳤습니다. **병합은 Day2(A100)** 에서 수행합니다 — 코랩 RAM 부담 없이, *"병합본을 또 받는"* 왕복도 없앱니다.

### 다음 교시 — Day1-6 · 도전: T4로 8B QLoRA · 정리
8B를 4bit로 올려 T4 한계를 시험하고(batch1·해상도↓), 4B vs 8B 트레이드오프를 표로 정리한 뒤 Day2(AWQ) 흐름을 예고합니다.

> ✅ **체크포인트**: ① Relaxed Accuracy를 설명할 수 있다 ② 학습 후 점수가 올랐다 ③ 어댑터만 올리고 **병합을 Day2로 미룬 이유(코랩 RAM)** 를 안다 ④ Hub에 어댑터가 올라갔다.